In [5]:
import pandas as pd
import numpy as np

# Common Preprocessing — Rossmann Store Sales

In [6]:
df = pd.read_csv("../data/train.csv")

df.head()

/var/folders/xd/fgpxnmwx71n9tz4rn52jp45h0000gn/T/ipykernel_45018/4220232724.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/train.csv")


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [7]:
df.shape

(1017209, 9)

In [8]:
df.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday'],
      dtype='object')

## PRE-PROCESSING

In [9]:
df = df.sort_values(["Store", "Date"]).reset_index(drop=True)

In [10]:
df[["Store", "Date", "Sales"]].head(20)

,Store,Date,Sales
0,1,2013-01-01,0
1,1,2013-01-02,5530
2,1,2013-01-03,4327
3,1,2013-01-04,4486
4,1,2013-01-05,4997
5,1,2013-01-06,0
6,1,2013-01-07,7176
7,1,2013-01-08,5580
8,1,2013-01-09,5471
9,1,2013-01-10,4892


In [11]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.isnull().sum())

(1017209, 9)
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']
Store             int64
DayOfWeek         int64
Date             object
Sales             int64
Customers         int64
Open              int64
Promo             int64
StateHoliday     object
SchoolHoliday     int64
dtype: object
Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64


In [12]:
print("Number of stores:", df["Store"].nunique())

print("Date range:")
print(df["Date"].min(), "to", df["Date"].max())

print("\nOpen values:")
print(df["Open"].value_counts())

print("\nPromo values:")
print(df["Promo"].value_counts())

print("\nStateHoliday values:")
print(df["StateHoliday"].value_counts())

print("\nDayOfWeek values:")
print(df["DayOfWeek"].value_counts().sort_index())

Number of stores: 1115
Date range:
2013-01-01 to 2015-07-31

Open values:
Open
1    844392
0    172817
Name: count, dtype: int64

Promo values:
Promo
0    629129
1    388080
Name: count, dtype: int64

StateHoliday values:
StateHoliday
0    855087
0    131072
a     20260
b      6690
c      4100
Name: count, dtype: int64

DayOfWeek values:
DayOfWeek
1    144730
2    145664
3    145665
4    145845
5    145845
6    144730
7    144730
Name: count, dtype: int64


In [13]:
# Make a copy so we don't accidentally modify the original dataframe
data = df.copy()

# Make sure Date is datetime
data["Date"] = pd.to_datetime(data["Date"])

# Clean StateHoliday
data["StateHoliday"] = data["StateHoliday"].astype(str)

# Convert categorical features to string
data["Store"] = data["Store"].astype(str)
data["DayOfWeek"] = data["DayOfWeek"].astype(str)
data["Promo"] = data["Promo"].astype(str)
data["SchoolHoliday"] = data["SchoolHoliday"].astype(str)

# Sort by store and date
data = data.sort_values(["Store", "Date"]).reset_index(drop=True)

# Create continuous time index
data["time_idx"] = (
    data["Date"] - data["Date"].min()
).dt.days

# Check
print(data.head())
print(data.dtypes)

  Store DayOfWeek       Date  Sales  Customers  Open Promo StateHoliday  \
0     1         2 2013-01-01      0          0     0     0            a   
1     1         3 2013-01-02   5530        668     1     0            0   
2     1         4 2013-01-03   4327        578     1     0            0   
3     1         5 2013-01-04   4486        619     1     0            0   
4     1         6 2013-01-05   4997        635     1     0            0   

  SchoolHoliday  time_idx  
0             1         0  
1             1         1  
2             1         2  
3             1         3  
4             1         4  
Store                    object
DayOfWeek                object
Date             datetime64[ns]
Sales                     int64
Customers                 int64
Open                      int64
Promo                    object
StateHoliday             object
SchoolHoliday            object
time_idx                  int64
dtype: object


In [14]:
print(data[["Date", "time_idx"]].head(20))

         Date  time_idx
0  2013-01-01         0
1  2013-01-02         1
2  2013-01-03         2
3  2013-01-04         3
4  2013-01-05         4
5  2013-01-06         5
6  2013-01-07         6
7  2013-01-08         7
8  2013-01-09         8
9  2013-01-10         9
10 2013-01-11        10
11 2013-01-12        11
12 2013-01-13        12
13 2013-01-14        13
14 2013-01-15        14
15 2013-01-16        15
16 2013-01-17        16
17 2013-01-18        17
18 2013-01-19        18
19 2013-01-20        19


## Train-Validation Split

A chronological train-validation split is used to prevent look-ahead data leakage. The final 60 days of the dataset are reserved for validation, while earlier observations are used for training.

In [15]:
max_date = data["Date"].max()

validation_start = max_date - pd.Timedelta(days=60)

train_data = data[data["Date"] < validation_start].copy()
val_data = data[data["Date"] >= validation_start].copy()

print("Training:")
print(train_data["Date"].min(), "to", train_data["Date"].max())
print("Rows:", len(train_data))

print("\nValidation:")
print(val_data["Date"].min(), "to", val_data["Date"].max())
print("Rows:", len(val_data))

Training:
2013-01-01 00:00:00 to 2015-05-31 00:00:00
Rows: 949194

Validation:
2015-06-01 00:00:00 to 2015-07-31 00:00:00
Rows: 68015


In [16]:
print("Training stores:", train_data["Store"].nunique())
print("Validation stores:", val_data["Store"].nunique())

print("\nTraining max date:", train_data["Date"].max())
print("Validation min date:", val_data["Date"].min())

Training stores: 1115
Validation stores: 1115

Training max date: 2015-05-31 00:00:00
Validation min date: 2015-06-01 00:00:00


## Save Common Processed Data

The full processed dataset and the common 60-day train/validation split are saved so all team members can use the same preprocessing.

In [17]:
data.to_csv("../data/processed/rossmann_processed.csv", index=False)
train_data.to_csv("../data/processed/rossmann_train.csv", index=False)
val_data.to_csv("../data/processed/rossmann_validation.csv", index=False)

print("Saved: rossmann_processed.csv")
print("Saved: rossmann_train.csv")
print("Saved: rossmann_validation.csv")

Saved: rossmann_processed.csv
Saved: rossmann_train.csv
Saved: rossmann_validation.csv
